In [ ]:
import matplotlib.pyplot as plt
import cv2
import time
import numpy as np
from IPython.display import display, Image, clear_output
import ipywidgets as widgets
import threading
import tensorflow as tf
import keras
import os
import random

In [ ]:
cap = cv2.VideoCapture(0)
cap.set(3,640) # adjust width
cap.set(4,480) # adjust height

cnames = ['bird', 'monkey', 'boar', 'tiger', 'rat', 'ram', 'dog', 'horse', 'hare', 'ox', 'dragon', 'snake']
model_name = "91porciento.keras"
IMAGE_SIZE = 300

def get_random_images_from_subfolders(base_path):
    images = {}
    for subfolder in os.listdir(base_path):
        subfolder_path = os.path.join(base_path, subfolder)
        if os.path.isdir(subfolder_path):
            files = [f for f in os.listdir(subfolder_path) if f.endswith('.jpg') or f.endswith('.png')]
            if files:
                random_file = random.choice(files)
                images[subfolder] = os.path.join(subfolder_path, random_file)
    print(images)
    return images

random_images = get_random_images_from_subfolders("dataset del chino")


# Cargar modelo (ya no necesitas definir la función preprocess)
model = tf.keras.models.load_model(model_name)

stop_event = threading.Event()

def classify_and_display():
    while not stop_event.is_set():
        ret, frame = cap.read()
        if not ret:
            continue

        # Preprocess frame for model
        img = cv2.resize(frame, (IMAGE_SIZE, IMAGE_SIZE))
        img = np.expand_dims(img, axis=0)

        # Predict class
        pred = model.predict(img)
        class_idx = np.argmax(pred)
        class_name = cnames[class_idx]

        # Load random image of predicted class
        class_img_path = random_images.get(class_name)
        if class_img_path:
            class_img = cv2.imread(class_img_path)
            class_img = cv2.resize(class_img, (frame.shape[1], frame.shape[0]))
        else:
            class_img = np.zeros_like(frame)

        # Concatenate images side by side
        combined = np.concatenate((frame, class_img), axis=1)
        _, jpeg = cv2.imencode('.jpg', combined)
        clear_output(wait=True)
        display(Image(data=jpeg.tobytes()))
        print(f"Predicted: {class_name}")

btn_stop = widgets.Button(description="Detener")
display(btn_stop)

def on_stop_clicked(b):
    stop_event.set()
    cap.release()
    clear_output(wait=True)
    print("Programa detenido.")

btn_stop.on_click(on_stop_clicked)

thread = threading.Thread(target=classify_and_display)
thread.start()